In [ ]:
import pandas as pd
import torch
import os
torch.cuda.device_count = lambda: 1

from transformers import PegasusTokenizer, PegasusForConditionalGeneration, Seq2SeqTrainer, Seq2SeqTrainingArguments
from datasets import Dataset
data_path = "/kaggle/input/datasets/subhan1501/text-summarizatio/arxiv_sample.csv"

print("Loading dataset...")
df = pd.read_csv(data_path, nrows=75000) 
df = df.dropna(subset=['abstract', 'title'])
print(f"Loaded {len(df)} clean rows for Pegasus training.")
model_name = "google/pegasus-xsum"
print(f"Downloading {model_name}...")

tokenizer = PegasusTokenizer.from_pretrained(model_name)
model = PegasusForConditionalGeneration.from_pretrained(model_name)
model.config.use_cache = False 

dataset = Dataset.from_pandas(df)

def tokenize_function(examples):
    model_inputs = tokenizer(examples["abstract"], max_length=128, truncation=True, padding="max_length")
    labels = tokenizer(examples["title"], max_length=32, truncation=True, padding="max_length")
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing dataset for Pegasus...")
tokenized_datasets = dataset.map(tokenize_function, batched=True)
save_path = "../models/Model_Pegasus"
os.makedirs(f"{save_path}/checkpoints", exist_ok=True)

torch.cuda.empty_cache()
print("Hardware Check: Forcing Single GPU Mode on T4")

args = Seq2SeqTrainingArguments(
    output_dir=f"{save_path}/checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=1,    
    gradient_accumulation_steps=4,    
    gradient_checkpointing=True,     
    fp16=True,                         
    optim="adafactor",                
    weight_decay=0.01,
    save_strategy="epoch",             
    save_total_limit=1,
    num_train_epochs=1,                
    logging_steps=100,
    eval_strategy="no",
)

trainer = Seq2SeqTrainer(
    model=model,
    args=args,
    train_dataset=tokenized_datasets,
    processing_class=tokenizer,
)

print("🔥 Starting Pegasus GPU Training Loop (Single T4 Forced)...")
trainer.train()
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Pegasus Training complete! Model securely saved to {save_path}")

Loading dataset...
Loaded 75000 clean rows for Pegasus training.


tokenizer_config.json:   0%|          | 0.00/87.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/1.91M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

...

🔥 Starting Pegasus GPU Training Loop (Single T4 Forced)...


Step,Training Loss
100,31.762561
200,29.103018
300,28.974512
400,27.206777
500,25.625015
600,21.717400
700,16.475219
800,9.934029
900,7.269385
1000,6.322413


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Pegasus Training complete! Model securely saved to ../models/Model_Pegasus
